# Day 5 — Retail Sales Exploratory Data Analysis

This notebook explores a **synthetic** retail sales dataset. All records are fictional and intended for portfolio practice.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "retail_sales.csv").exists() and (ROOT.parent / "data" / "retail_sales.csv").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "retail_sales.csv", parse_dates=["order_date"])
df.head()

## Inspect the dataset

In [ ]:
print("Shape:", df.shape)
df.info()
display(df.describe(numeric_only=True))
display(df.isna().sum())

## Key business metrics

In [ ]:
df["month"] = df["order_date"].dt.to_period("M").astype(str)
summary = {
    "rows": len(df),
    "unique_orders": df["order_id"].nunique(),
    "total_units": int(df["quantity"].sum()),
    "gross_sales_inr": round(df["gross_sales_inr"].sum(), 2),
    "net_sales_inr": round(df["net_sales_inr"].sum(), 2),
    "average_order_value_inr": round(df.groupby("order_id")["net_sales_inr"].sum().mean(), 2),
    "top_category": df.groupby("category")["net_sales_inr"].sum().idxmax(),
    "top_city": df.groupby("city")["net_sales_inr"].sum().idxmax(),
}
pd.Series(summary, name="value")

## Monthly net sales trend

In [ ]:
sns.set_theme(style="whitegrid")
monthly = df.groupby("month", as_index=False)["net_sales_inr"].sum()
plt.figure(figsize=(11, 5))
sns.lineplot(data=monthly, x="month", y="net_sales_inr", marker="o")
plt.title("Monthly Net Sales Trend")
plt.xlabel("Month")
plt.ylabel("Net sales (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Category and city performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
category = df.groupby("category", as_index=False)["net_sales_inr"].sum().sort_values("net_sales_inr", ascending=False)
city = df.groupby("city", as_index=False)["net_sales_inr"].sum().sort_values("net_sales_inr", ascending=False)
sns.barplot(data=category, x="category", y="net_sales_inr", ax=axes[0])
sns.barplot(data=city, x="city", y="net_sales_inr", ax=axes[1])
axes[0].set_title("Net Sales by Category")
axes[1].set_title("Net Sales by City")
axes[0].tick_params(axis="x", rotation=25)
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Payment method mix

In [ ]:
payments = df["payment_method"].value_counts().rename_axis("payment_method").reset_index(name="orders")
sns.barplot(data=payments, x="payment_method", y="orders")
plt.title("Orders by Payment Method")
plt.xlabel("Payment method")
plt.ylabel("Orders")
plt.tight_layout()
plt.show()

## Interpretation

Use the tables and charts to identify the top category and city, compare monthly sales, and review payment method usage. Since the data is synthetic, treat results as practice examples, not real market evidence.